## Error Decomposition

- Input: $z_{pred} − z_{target}$
- Tools: SAE, PCA on errors, magnitude analysis

*"what does the model systematically get wrong, and is it clinically structured?"*

In [ ]:

import numpy as np
from pathlib import Path

from src.analysis.eval_infra import (
    load_label, 
    load_escalation_labels, 
    compute_escalation_criterions)
from src.utils.io import (EXPERIMENTS_DIR, DATA_DIR, 
                          load_embeddings, load_sequences_dict, 
                          load_metadata, load_json, load_npz_dict)
from src.utils.seed import load_exp_seed, set_global_seed

In [ ]:
# -- Config Settings --
MODEL       = "test_01"
EMB_NAME    = "embeddings_40.npz"

In [ ]:
EXPERIMENTS     = Path("experiments")
MODEL_DIR       = EXPERIMENTS / MODEL
ANALYSIS_DIR    = MODEL_DIR / "analysis" / "error_decomp"
FIGURES_DIR     = ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCES_PATH  = Path(DATA_DIR / "sequences.jsonl")

set_global_seed(load_exp_seed(EXPERIMENTS_DIR / MODEL))

# -- Load embeddings --
emb, EMB_PATH = load_embeddings(MODEL_DIR, EMB_NAME)
print(f"Embeddings: {EMB_PATH.name}")

z_encs = emb["z_encs"]             # (N, C_padded, D)
z_pred = emb["z_pred"]             # (N, D)
z_target = emb["z_target"]         # (N, D)
ctx_pad_mask = emb["ctx_pad_mask"]  # (N, C_padded)
subject_ids = emb["subject_ids"]   # (N,)
mask_pos = emb["mask_pos"]         # (N,)
pred_error = z_pred - z_target     # (N, D)

# Flatten valid encounters from z_encs
valid = ~ctx_pad_mask.astype(bool)
z_enc_flat = z_encs[valid]         # (N_valid, D)
enc_subject_ids = np.broadcast_to(
    subject_ids[:, None], ctx_pad_mask.shape)[valid]
enc_positions = np.broadcast_to(
    np.arange(ctx_pad_mask.shape[1])[None, :], ctx_pad_mask.shape)[valid]

print(f"  z_encs:      {z_encs.shape}")
print(f"  z_enc_flat:  {z_enc_flat.shape}")
print(f"  z_pred:      {z_pred.shape}")
print(f"  z_target:    {z_target.shape}")
print(f"  pred_error:  {pred_error.shape}")
print(f"  subjects:    {len(np.unique(subject_ids))}")

# -- Load labels from sequences.jsonl
patients = load_sequences_dict(SEQUENCES_PATH)
unique_sids = np.unique(subject_ids)

label_escalation_patient = load_label(patients, unique_sids, "label_escalation")
label_30d_patient = load_label(patients, unique_sids, "label_30d")
label_esc_per_sample = load_escalation_labels(patients, subject_ids, mask_pos)
criteria_labels = compute_escalation_criterions(patients, subject_ids, mask_pos)
 
# Sample-level encounter labels (for the masked encounter)
label_esc_per_sample = np.array([
    patients[str(sid)]["label_escalation_per_enc"][int(mp)]
    if "label_escalation_per_enc" in patients[str(sid)] else 0
    for sid, mp in zip(subject_ids, mask_pos)])
 
print(f"  Escalation rate (patient): {label_escalation_patient.mean():.3f}")
print(f"  Escalation rate (encounter): {label_esc_per_sample.mean():.3f}")
print(f"  30d readmit rate: {label_30d_patient.mean():.3f}")

In [ ]:
try:
    meta, meta_names, meta_pids = load_metadata(DATA_DIR)
    print(f"  Metadata: {meta.shape[0]} patients x {meta.shape[1]} features")
except FileNotFoundError:
    print("  Metadata not found")

In [ ]:
results = load_json(ANALYSIS_DIR / "representation.json")
if results is None:
    raise FileNotFoundError(f"No predictor results found")
error_pca   = load_npz_dict(ANALYSIS_DIR / "error_pca.npz")
error_umap  = np.load(ANALYSIS_DIR / "error_umap.npy")
sae_data    = load_json(ANALYSIS_DIR / "sae_features.json")
crossref    = load_json(ANALYSIS_DIR / "sae_crossref.json")
s1_pca_enc  = np.load(MODEL_DIR / "analysis" / "representation" / "pca_encounter.npz")

## Plotting

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

**Error eigenvalue spectrum overlay**: two lines on the same plot: $z_{enc}$ eigenvalues (from Stage 1) and pred_error eigenvalues. Both normalized to sum-to-1 for comparison. 

Annotate effective dimensionality of each. If error has a sharper elbow, annotate that - it means errors are more structured than representations.

In [ ]:
from src.analysis.plotting import _s3_eigenvalue_overlay
_s3_eigenvalue_overlay(results, error_pca, s1_pca_enc,
                       show=True, save=False, fig_dir=FIGURES_DIR)

**Magnitude distribution** overlapping histograms of $\|pred_error\|$ for escalation=1 vs escalation=0. Annotate Mann-Whitney U statistic and p-value. Optionally, a second panel splitting by escalation criterion.

In [ ]:
from src.analysis.plotting import _s3_magnitude_distribution
_s3_magnitude_distribution(results,
                           show=True, save=False, fig_dir=FIGURES_DIR)

**Error UMAP** 2D scatter colored by escalation type (same coloring as Stage 1 UMAP panel c). If error clusters correspond to escalation criteria, that's a main finding.

In [ ]:
if error_umap is not None:
    from src.analysis.plotting import _s3_error_umap
    _s3_error_umap(error_umap, [0,1], criteria_labels,
                   show=True, save=False, fig_dir=FIGURES_DIR)

**Critical figure**: **SAE cross-reference heatmap** (if crossref results exist):
- Rows = z_enc SAE features
- columns = pred_error SAE features
- Cell color = cosine similarity of decoder directions
- Highlight cells above 0.8 threshold. 

Annotate summary: "X% of error features align with encoder features." 

This is the figure that answers "does the model fail along the same dimensions it represents?"

In [ ]:
if crossref:
    from src.analysis.plotting import _s3_sae_crossref_heatmap
    _s3_sae_crossref_heatmap(crossref,
                             show=True, save=False, fig_dir=FIGURES_DIR)

**SAE feature cards for prediction error $z_{pred}-z_{target}$** (if SAE results exist): same format as Stage 1 cards but for prediction error features. Each card shows what clinical patterns the model systematically gets wrong.

In [ ]:
if sae_data and sae_data.get("feature_cards"):
    from src.analysis.plotting import _s3_sae_feature_cards
    _s3_sae_feature_cards(sae_data, top_n=12,
                          show=True, save=False, fig_dir=FIGURES_DIR)